# 09 — Staffing / Counter Recommendation

Turns forecast demand + observed service rates + the executive's wait target into **how many
counters to open each hour**, using Erlang-C (M/M/c) queueing. Replaces the old
"avg wait ≥ 20 → review staffing" rule with an actionable hourly plan.

**Outputs the `staffing_recommendation` insight** consumed by the Manager dashboard.

## 1. Setup & data

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams.update({"axes.titleweight": "bold", "axes.titlesize": 12, "figure.dpi": 110})
pd.set_option("display.max_columns", 40)

# QMe Now palette (matches the admin dashboards)
NAVY, STEEL, TEAL, RED, GOLD = "#2F5063", "#6E8AA6", "#2E7387", "#B23A4E", "#9A6B2E"
BLUES = sns.light_palette(NAVY, n_colors=6, reverse=True)

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE))
DOW = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
def hour_label(h): return f"{((int(h) + 11) % 12) + 1}{'am' if h < 12 else 'pm'}"
print("Ready.")

In [ ]:
from scripts import recommend_staffing as rs

conn = rs.connect()
df, counters, targets = rs.load_data(conn)
print(f"{len(df):,} visit rows · {df.branch_name.nunique()} branches · {df.service_id.nunique()} services")
df[["branch_name", "service_name", "hour", "service_time_minutes"]].head()

## 2. The queueing curve\nErlang-C: for a given arrival rate and service rate, adding counters cuts the expected wait sharply — then flattens. The model picks the smallest count that meets the target.

In [ ]:
lam, mu, target = 18.0, 4.0, 20  # arrivals/hr, served/hr/counter, target wait (min)
cs = list(range(1, 11))
waits = [rs.erlang_c_wait_minutes(lam, mu, c) for c in cs]
waits = [w if np.isfinite(w) else np.nan for w in waits]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cs, waits, color=NAVY, lw=2.6, marker="o")
ax.axhline(target, color=RED, ls="--", lw=1.6, label=f"Target {target} min")
rec, _ = rs.recommend_servers(lam, mu, target, 10)
ax.axvline(rec, color=TEAL, ls=":", lw=1.8, label=f"Recommended: {rec} counters")
ax.set_title(f"Expected Wait vs Counters Open (λ={lam}/hr, μ={mu}/hr)")
ax.set_xlabel("Counters open"); ax.set_ylabel("Expected wait (min)"); ax.set_ylim(0, 60); ax.legend()
plt.tight_layout(); plt.show()

## 3. Hourly staffing plan for a branch

In [ ]:
insights, _, _ = rs.build_insights(df, counters, targets)
branch = insights[0]["insight_data"]["branches"][0]
plan = pd.DataFrame(branch["hourly_plan"])
plan["hour"] = plan["hour"].map(hour_label)
print(f"{branch['branch_name']} · target {branch['target_wait_minutes']} min · {branch['available_counters']} counters available")
plan[["hour", "recommended_counters", "unconstrained_counters", "available_counters", "expected_wait_minutes", "over_capacity"]]

In [ ]:
p = pd.DataFrame(branch["hourly_plan"])
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(p))
ax.bar(x - 0.2, p["recommended_counters"], 0.4, label="Recommended", color=NAVY)
ax.bar(x + 0.2, p["available_counters"], 0.4, label="Available", color=STEEL)
over = p[p["over_capacity"]]
if len(over):
    ax.scatter(over.index, over["available_counters"] + 0.3, color=RED, marker="v", s=60, label="Over capacity", zorder=5)
ax.set_xticks(x); ax.set_xticklabels([hour_label(h) for h in p["hour"]])
ax.set_title(f"Counters Needed by Hour — {branch['branch_name']}"); ax.set_ylabel("Counters"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
if os.getenv("WRITE_DB") == "1":
    ins, gen, stale = rs.build_insights(df, counters, targets)
    rs.upsert_insights(conn, ins, gen, stale, rs.MODEL_VERSION)
    print(f"Upserted {len(ins)} staffing_recommendation insight(s).")
else:
    print("Preview only — set WRITE_DB=1 to persist.")
conn.close()